# Multi-Scale Conformer & Offline RL 2-Stage Trading Pipeline (XAU/USD)

This notebook trains the full end-to-end quantitative trading pipeline for 5-minute Gold return forecasting and position execution:

- **Stage 1 (Market Predictor)**: Multi-Scale Causal Inception-Conformer with Macaron attention, causal 1D Inception convolutions, continuous RBF embeddings, and high-priority directional composite loss (Volatility-Weighted Directional Focal Loss $\mathcal{L}_{\text{focal}}$ + Distribution Expectation Sign Alignment $\mathcal{L}_{\text{dir\_sign}}$ + Wrong-Mass Penalty $\mathcal{L}_{\text{dir\_mass}}$ + Soft-CE + Direction Gate + Huber).
- **Stage 1.5 (Institutional Causal Audit)**: Strict out-of-sample coverage performance breakdown (100%, 50%, 30%, 20%, 10% coverage thresholds), Information Coefficient, and causal placebo verifications (feature shuffling, +/-1 bar temporal shift).
- **Stage 2 (Execution Policy)**: Offline Contextual Bandit RL Policy mapping the frozen 31-dim Conformer state vector to 7 discrete position sizing actions with asymmetric directional reward multipliers ($1.5\times$ correct direction reward + discovery bonus, $2.0\times$ counter-trend penalty).

> **GPU Setup:** Ensure GPU accelerator (**GPU T4 x2**) is enabled in the Kaggle notebook settings (Right Sidebar -> Accelerator -> GPU).

In [ ]:
# 1. GPU Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("WARNING: Running on CPU! Switch Accelerator to 'GPU T4 x2' in Kaggle right sidebar.")
!nvidia-smi

In [ ]:
# 2. Workspace Setup, Code & Config Extraction
import os
import glob
import zipfile
import shutil
from pathlib import Path

# 1. Create target directories in /kaggle/working
for d in ["/kaggle/working/src", "/kaggle/working/configs", "/kaggle/working/data/processed", "/kaggle/working/data/models", "/kaggle/working/data/reports"]:
    Path(d).mkdir(parents=True, exist_ok=True)

# 2. Inspect /kaggle/input
print("Attached files in /kaggle/input:")
for f in glob.glob("/kaggle/input/**/*", recursive=True):
    print(" -", f)

# 3. Unpack code zip if present
zip_matches = glob.glob("/kaggle/input/**/kaggle_code.zip", recursive=True)
if zip_matches:
    with zipfile.ZipFile(zip_matches[0], 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")
    print(f"\nSuccessfully extracted code from: {zip_matches[0]}")

# 4. Copy any loose python scripts and yaml configs from /kaggle/input (handles Kaggle auto-unzip)
for py_file in glob.glob("/kaggle/input/**/*.py", recursive=True):
    shutil.copy(py_file, "/kaggle/working/src/")

for yaml_file in glob.glob("/kaggle/input/**/*.yaml", recursive=True) + glob.glob("/kaggle/input/**/*.yml", recursive=True):
    shutil.copy(yaml_file, "/kaggle/working/configs/")

# 5. Ensure default configs exist (Fail-safe)
default_cfg = Path("/kaggle/working/configs/gold_continuous_45m.yaml")
if not default_cfg.exists():
    default_cfg.write_text("""symbol: XAUUSD
raw_1m_path: data/raw/xauusd_1m_continuous.parquet
model_dataset: gold_multiscale_45m_dataset.npz
lookback_minutes: 45
gap_minutes: 5
num_rbf_bins: 9
rbf_sigma: 0.5
ewma_alpha: 0.06
cost: 0.50
label_quantiles:
  - 0.00
  - 0.05
  - 0.15
  - 0.30
  - 0.45
  - 0.55
  - 0.70
  - 0.85
  - 0.95
  - 1.00
""")
    print("Created default config: configs/gold_continuous_45m.yaml")

# 6. Link dataset .npz into data/processed/
npz_matches = glob.glob("/kaggle/input/**/gold_multiscale_45m_dataset.npz", recursive=True)
if npz_matches:
    dest_npz = Path("/kaggle/working/data/processed/gold_multiscale_45m_dataset.npz")
    if not dest_npz.exists():
        os.symlink(npz_matches[0], dest_npz)
    print(f"Successfully linked dataset from: {npz_matches[0]} -> {dest_npz}")
else:
    print("\nWARNING: gold_multiscale_45m_dataset.npz not found in /kaggle/input! Checking for raw 1m parquet...")
    parquet_matches = glob.glob("/kaggle/input/**/xauusd_1m_continuous.parquet", recursive=True)
    if parquet_matches:
        print(f"Found raw parquet at {parquet_matches[0]}. Building continuous dataset...")
        !python /kaggle/working/src/build_continuous_multiscale_dataset.py --config /kaggle/working/configs/gold_continuous_45m.yaml
    else:
        print("Make sure dataset is attached to notebook inputs.")

# 7. Verify /kaggle/working contents
print("\nFiles in /kaggle/working/src/:", os.listdir("/kaggle/working/src"))
print("Files in /kaggle/working/configs/:", os.listdir("/kaggle/working/configs"))

In [ ]:
# 3. Verify Dataset Integrity
import numpy as np

data_path = "/kaggle/working/data/processed/gold_multiscale_45m_dataset.npz"
if os.path.exists(data_path):
    data = np.load(data_path)
    print("=== Dataset Integrity Verification ===")
    print(f"Total Examples:        {len(data['cause_windows']):,}")
    print(f"Cause Windows Tensor:  {data['cause_windows'].shape} (N, 45 bars, 28 features)")
    print(f"Group Features Tensor: {data['group_features'].shape} (N, 12 transition metrics)")
    print(f"Continuous Targets:    {data['targets'].shape} (5-min price delta)")
    print(f"Target Quantile Edges: {np.round(data['target_edges'], 4).tolist()}")
else:
    print(f"Dataset not found at {data_path}!")

## Stage 1: Train Causal Inception-Conformer Market Predictor
Trains the multi-task backbone with causal attention masking and dominant directional discovery loss (Volatility-Weighted Directional Focal Loss $\mathcal{L}_{\text{focal}}$, Distribution Expectation Sign Alignment $\mathcal{L}_{\text{dir\_sign}}$, Asymmetric Wrong-Mass Penalty $\mathcal{L}_{\text{dir\_mass}}$, Soft-CE, RPS, Huber, and Direction Gate).

In [ ]:
# 4. Stage 1 Execution
import sys
for p in ["/kaggle/working/src", "/kaggle/working"]:
    if p not in sys.path:
        sys.path.append(p)

!python /kaggle/working/src/train_multiscale_conformer.py \
    --config /kaggle/working/configs/gold_continuous_45m.yaml \
    --epochs 5 \
    --batch-size 512 \
    --lr 1e-3

## Stage 1.5: Institutional Causal Audit & 30% Coverage Performance
Evaluates out-of-sample directional performance across conviction cutoffs (including Top 30% Coverage win rate) and executes zero-lookahead placebo tests (feature shuffling & +/- 1 bar temporal shifts).

In [ ]:
# 4.5 Stage 1.5 Execution (Institutional Audit)
!python /kaggle/working/src/audit_conformer.py \
    --config /kaggle/working/configs/gold_continuous_45m.yaml \
    --checkpoint /kaggle/working/data/models/gold_multiscale_conformer_oos.pt

## Stage 2: Train Offline RL Execution & Position Sizing Policy
Extracts 31-dim multi-task state vectors from the **frozen Stage 1 Conformer** checkpoint and trains an offline contextual bandit policy directly maximizing net PnL across 7 position sizes with asymmetric directional reward multipliers ($1.5\times$ correct direction reward + discovery bonus, $2.0\times$ counter-trend penalty).

In [ ]:
# 5. Stage 2 Execution
!python /kaggle/working/src/train_multiscale_rl_policy.py \
    --config /kaggle/working/configs/gold_continuous_45m.yaml \
    --epochs 10 \
    --cost 0.50 \
    --lr 1e-3

In [ ]:
# 6. Package Trained Checkpoints, Reports & Logs for Download
!zip -r /kaggle/working/gold_multiscale_trading_system_results.zip \
    /kaggle/working/data/models \
    /kaggle/working/data/reports

print("\n[SUCCESS] Pipeline Completed!")
print("Download 'gold_multiscale_trading_system_results.zip' from the Output section in the right sidebar.")